# Stage 4 — Build the Supervised Dataset

Turn three gridded cubes (CHL, SST, bathy) into a tidy table of `(features at time t) → (CHL at t+h)`, keeping gaps as `NaN` and respecting the no-shuffling-time rule.

## Why this is the riskiest stage in the whole project
Most silent bugs in time-series ML are born here:
- **Off-by-one shifts** between features and targets → leakage → impressive-looking R² that is pure noise.
- **Mixed lag direction** (lags going forward, or targets going backward) → same outcome.
- **Coord misalignment** between predictors and target → features paired with the wrong pixel.
- **Accidentally filling NaNs** when constructing the target → invalidates the methodological core.

The validation gate at the end runs a per-pixel **leakage assert** that catches all four. **Do not skip the gate.** PLAN.md: *"The single most important check in the whole project is the leakage assert."*

## What this notebook produces
- A parquet file `data/processed/supervised_sample.parquet`
- One row per (pixel × time) sampled to keep memory tractable
- Columns: `time, latitude, longitude, <features…>, target_h1 … target_h7`
- Gaps preserved as NaN — XGBoost will use them natively in Stage 6

## 4.0 — Configuration

We respect the **coverage decision** from Notebook 2. If Gate 2 said `3-DAY COMPOSITE`, the target is wrapped in a 3-day rolling mean; otherwise daily.

In [2]:
from pathlib import Path
import xarray as xr
import numpy as np

RAW_DIR  = Path("data/raw")
PROC_DIR = Path("data/processed")
PROC_DIR.mkdir(parents=True, exist_ok=True)

CHL_PATH   = RAW_DIR / "chl_malta_2021_2023.nc"
SST_PATH   = RAW_DIR / "sst_malta_2021_2023.nc"
BATHY_PATH = RAW_DIR / "bathy_malta.nc"
OUT_PATH   = PROC_DIR / "supervised_sample.parquet"

for p in (CHL_PATH, SST_PATH, BATHY_PATH):
    assert p.exists(), f"Missing {p} — run earlier notebooks first."

# Forecast horizons (days)
HORIZONS   = [1, 2, 3, 4, 5, 6, 7]
LAGS       = [1, 2, 3, 7, 14]
ROLL_WIN   = 7

# Sampling — total rows ≈ SAMPLE_PER_MONTH × 12 × n_years.
# 10k × 36 months ≈ 360k rows; tractable in memory, plenty for XGBoost.
SAMPLE_PER_MONTH = 10_000
SEED             = 42

# Read the Gate 2 decision so the target shape matches the choice we made.
decision_file = PROC_DIR / "coverage_decision.txt"
if decision_file.exists():
    decision_line = decision_file.read_text().splitlines()[0].strip()
    USE_3DAY_TARGET = decision_line == "3-DAY COMPOSITE"
else:
    USE_3DAY_TARGET = False
    print("Warning: no coverage_decision.txt found — assuming DAILY target.")

print(f"Target mode      : {'3-day rolling composite' if USE_3DAY_TARGET else 'daily'}")
print(f"Horizons         : {HORIZONS}")
print(f"Lags (days)      : {LAGS}")
print(f"Rolling window   : {ROLL_WIN}")
print(f"Sample per month : {SAMPLE_PER_MONTH}")

Target mode      : daily
Horizons         : [1, 2, 3, 4, 5, 6, 7]
Lags (days)      : [1, 2, 3, 7, 14]
Rolling window   : 7
Sample per month : 10000


## 4.1 — Load cubes & align onto the CHL grid

**Rule:** the CHL grid is finest, so everything gets regridded *onto it*, never the other way around. Spatial interpolation of SST (~5 km → 1 km) does not invent real detail — it just makes the arrays addressable on a common grid. For a tree model this is fine; coarse predictors contribute large-scale signal, which is exactly what we want from them.

In [3]:
chl_ds   = xr.open_dataset(CHL_PATH)
sst_ds   = xr.open_dataset(SST_PATH)
bathy_ds = xr.open_dataset(BATHY_PATH)

chl = chl_ds["CHL"]                   # mg m-3, gappy
sst = sst_ds["analysed_sst"]          # kelvin, gap-free L4
elev_var = next(v for v in bathy_ds.data_vars if v in ("z", "elevation", "bed_elevation"))
elev = bathy_ds[elev_var]

# Convert SST K → °C (interpretable, and prevents sign-confusion later)
if float(sst.isel(time=0).max()) > 200:
    sst = sst - 273.15
    sst.attrs["units"] = "degC"
    print("SST converted K → °C")

# Bathy → 'depth' in metres, positive = deeper. Land becomes 0.
depth = -elev.where(elev < 0, 0)
depth.name = "depth_m"

# log10(CHL) is the standard error space for ocean colour.
log_chl = np.log10(chl).rename("log_chl")

# Spatial alignment onto the CHL grid (the finest grid).
# SST has 'latitude'/'longitude' already; if a future predictor uses lat/lon, rename first.
sst_on_chl   = sst.interp(latitude=chl.latitude, longitude=chl.longitude)
depth_on_chl = depth.interp(latitude=chl.latitude, longitude=chl.longitude)

# Predictor & target time alignment (predictors must exist on the same calendar)
sst_on_chl = sst_on_chl.reindex(time=chl.time, method="nearest", tolerance=np.timedelta64(1,"D"))

print(f"CHL    : {dict(chl.sizes)}")
print(f"SST    : {dict(sst_on_chl.sizes)}  (regridded)")
print(f"Depth  : {dict(depth_on_chl.sizes)}  (static)")

SST converted K → °C
CHL    : {'time': 1095, 'latitude': 198, 'longitude': 272}
SST    : {'time': 1095, 'latitude': 198, 'longitude': 272}  (regridded)
Depth  : {'latitude': 198, 'longitude': 272}  (static)


## 4.2 — Define the target series

If Gate 2 selected the 3-day composite, the target (and the `log_chl_t0` feature it derives from) is a 3-day rolling mean. The target horizons are built off this composited series; predictors stay daily.

In [4]:
if USE_3DAY_TARGET:
    target_series = log_chl.rolling(time=3, min_periods=1).mean().rename("log_chl")
    print("target = 3-day rolling mean of log10(CHL)")
else:
    target_series = log_chl
    print("target = daily log10(CHL)")

target = daily log10(CHL)


## 4.3 — Build features (lagged, rolling, gap-age, SST, static, temporal)

Five feature families:
1. **Autoregressive lags** — `log_chl` at t-1, t-2, t-3, t-7, t-14.
2. **Rolling statistics** — 7-day mean & std of `log_chl`.
3. **Gap-age** — days since last valid CHL observation, per pixel. Encodes *how stale* the last observation is.
4. **SST features** — t0, t-1, and 7-day rolling mean.
5. **Static + temporal** — bathymetric depth, day-of-year `sin`/`cos`.

All lags go **backward** (`shift(time=+lag)`); targets go **forward** (`shift(time=-h)`). Mixing those directions is the classic catastrophic bug — the gate catches it.

In [5]:
feat = xr.Dataset()
feat["log_chl_t0"] = target_series

# Autoregressive lags on the target series
for lag in LAGS:
    feat[f"log_chl_lag{lag}"] = target_series.shift(time=lag)

# Rolling stats on the target series
feat["log_chl_roll7_mean"] = target_series.rolling(time=ROLL_WIN, min_periods=1).mean()
feat["log_chl_roll7_std"]  = target_series.rolling(time=ROLL_WIN, min_periods=2).std()

# SST features
feat["sst_t0"]         = sst_on_chl
feat["sst_lag1"]       = sst_on_chl.shift(time=1)
feat["sst_roll7_mean"] = sst_on_chl.rolling(time=ROLL_WIN, min_periods=1).mean()

# Static — depth, broadcast across time so it joins cleanly
feat["depth_m"] = depth_on_chl.broadcast_like(target_series)

# Day-of-year cyclic encoding (so the model can pick up annual cycle if useful)
doy = target_series.time.dt.dayofyear
feat["doy_sin"] = np.sin(2*np.pi*doy/365.25).broadcast_like(target_series)
feat["doy_cos"] = np.cos(2*np.pi*doy/365.25).broadcast_like(target_series)

print("Feature variables built:")
for name in feat.data_vars:
    print(f"  {name}")

c:\GIT\mcast.pages.ai.v2\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Feature variables built:
  log_chl_t0
  log_chl_lag1
  log_chl_lag2
  log_chl_lag3
  log_chl_lag7
  log_chl_lag14
  log_chl_roll7_mean
  log_chl_roll7_std
  sst_t0
  sst_lag1
  sst_roll7_mean
  depth_m
  doy_sin
  doy_cos


## 4.4 — Gap-age feature

Per pixel, days since the last valid CHL observation. Implemented as a vectorised loop over the time axis on the underlying NumPy array — pure xarray `.shift` chains would explode in memory.

In [6]:
valid = (~np.isnan(log_chl.values)).astype(np.int32)   # 0/1, (time, lat, lon)
gap_age = np.empty_like(valid)
gap_age[0] = 0
for t in range(1, valid.shape[0]):
    gap_age[t] = np.where(valid[t], 0, gap_age[t-1] + 1)

feat["gap_age"] = xr.DataArray(
    gap_age, dims=log_chl.dims, coords=log_chl.coords, name="gap_age"
)

# Sanity print: gap_age is 0 wherever log_chl is valid
ga_valid = float(feat["gap_age"].where(~np.isnan(log_chl)).max())
ga_gap   = float(feat["gap_age"].where(np.isnan(log_chl)).max())
print(f"max gap_age on valid pixels: {ga_valid}  (expected 0)")
print(f"max gap_age on gap pixels  : {ga_gap}  (run-length of longest gap)")

max gap_age on valid pixels: 0.0  (expected 0)
max gap_age on gap pixels  : 1094.0  (run-length of longest gap)


## 4.5 — Forecast targets

One target per horizon. `shift(time=-h)` moves observations from `t+h` back to row `t`, so `target_h{h}` at row `t` literally is *the value at* `t+h`. That's the contract the gate will verify.

In [7]:
targets = xr.Dataset()
for h in HORIZONS:
    targets[f"target_h{h}"] = target_series.shift(time=-h)
print("Target variables:", list(targets.data_vars))

Target variables: ['target_h1', 'target_h2', 'target_h3', 'target_h4', 'target_h5', 'target_h6', 'target_h7']


## 4.6 — Flatten to a tidy DataFrame

`to_dataframe()` gives a long table indexed by `(time, latitude, longitude)`. We reset the index so all three become regular columns; the resulting `df` is what every later notebook trains on.

In [8]:
import pandas as pd

df_feat = feat.to_dataframe()
df_tgt  = targets.to_dataframe()
df = pd.concat([df_feat, df_tgt], axis=1).reset_index()

print(f"Full table shape: {df.shape}")
print(f"NaN fraction per column:")
print((df.isna().mean() * 100).round(1).rename("%NaN"))

Full table shape: (58972320, 25)
NaN fraction per column:
latitude               0.0
longitude              0.0
time                   0.0
log_chl_t0            51.4
log_chl_lag1          51.4
log_chl_lag2          51.4
log_chl_lag3          51.4
log_chl_lag7          51.5
log_chl_lag14         51.6
log_chl_roll7_mean    17.0
log_chl_roll7_std     22.3
sst_t0                17.5
sst_lag1              17.6
sst_roll7_mean        17.5
depth_m                0.0
doy_sin                0.0
doy_cos                0.0
gap_age                0.0
target_h1             51.4
target_h2             51.5
target_h3             51.5
target_h4             51.6
target_h5             51.6
target_h6             51.7
target_h7             51.7
Name: %NaN, dtype: float64


## 4.7 — Sample pixels (stratified by month)

The full table is ~96M rows (1095 days × ~87k pixels). We can't fit that in memory for XGBoost training, and we don't need to — pixels are highly spatially correlated. A **stratified-by-month** sample keeps seasonal balance while shrinking the table by ~100×.

Important: we drop rows where `log_chl_t0` is NaN (no current observation = no useful row). We do **not** drop rows where lag features are NaN — XGBoost handles those natively in Stage 6.

In [9]:
df["month"] = df["time"].dt.month

valid_rows = df.dropna(subset=["log_chl_t0"])
print(f"Rows with valid log_chl_t0: {len(valid_rows):,} of {len(df):,}")

# Explicit per-month loop — avoids the pandas-2.2 groupby.apply column-drop
# behaviour, and is clearer to read.
parts = []
for month, group in valid_rows.groupby("month"):
    n_take = min(len(group), SAMPLE_PER_MONTH)
    parts.append(group.sample(n_take, random_state=SEED))
sample = pd.concat(parts, ignore_index=True)

# Belt-and-braces: rebuild the month column from time so it's always present,
# regardless of how groupby chose to preserve (or drop) it.
sample["month"] = sample["time"].dt.month

print(f"Sampled table shape: {sample.shape}")
print(f"\nRows per month:")
print(sample["month"].value_counts().sort_index())

sample.to_parquet(OUT_PATH)
print(f"\nSaved → {OUT_PATH}  ({OUT_PATH.stat().st_size/1e6:.1f} MB)")

Rows with valid log_chl_t0: 28,674,395 of 58,972,320
Sampled table shape: (120000, 26)

Rows per month:
month
1     10000
2     10000
3     10000
4     10000
5     10000
6     10000
7     10000
8     10000
9     10000
10    10000
11    10000
12    10000
Name: count, dtype: int64

Saved → data\processed\supervised_sample.parquet  (12.2 MB)


---
## Validation Gate 4 — the leakage assert

This is the most important check in the whole project. The contract:

> For any pixel, the value of `target_h{h}` at time `t` must equal the value of `log_chl_t0` at time `t+h` for the same pixel.

We pick a few pixels with long valid runs, sort by time, and assert the shift. If lags went the wrong way, or targets were misaligned, this fires.

Five checks:
1. **Per-pixel target alignment** — the assert above.
2. **Lag direction** — `log_chl_lag1` at time t equals `log_chl_t0` at time t-1 (for the same pixel).
3. **NaN preservation** — lag columns retain NaN at gap pixels (we did not silently fill).
4. **Gap-age contract** — `gap_age == 0` exactly where `log_chl_t0` is valid.
5. **No fully-empty target column** — at least one valid value per horizon.

In [10]:
import pandas as pd

# Re-load the full (not sampled) feature/target cubes for the gate — we need
# consecutive timesteps for the same pixel, which sampling would destroy.
full = pd.concat([feat.to_dataframe(), targets.to_dataframe()], axis=1).reset_index()

results = {}

# Pick three pixels with the most valid observations to make the assert meaningful
pixel_validity = (full.dropna(subset=["log_chl_t0"])
                      .groupby(["latitude", "longitude"])
                      .size()
                      .sort_values(ascending=False))
test_pixels = pixel_validity.head(3).index.tolist()

# --- 1) per-pixel target alignment for each horizon ---
align_ok = True
for (lat, lon) in test_pixels:
    p = full[(full.latitude == lat) & (full.longitude == lon)].sort_values("time").reset_index(drop=True)
    for h in HORIZONS:
        col = f"target_h{h}"
        # target_h{h}[t] should equal log_chl_t0[t+h]
        lhs = p[col].iloc[:-h].values
        rhs = p["log_chl_t0"].iloc[h:].values
        # allclose with equal_nan=True so gappy pixels still pass
        if not np.allclose(lhs, rhs, equal_nan=True):
            align_ok = False
            print(f"  ALIGN FAIL pixel=({lat:.3f},{lon:.3f}) h={h}")
results["per-pixel target alignment (all horizons)"] = align_ok

# --- 2) lag direction: log_chl_lag1[t] == log_chl_t0[t-1] ---
lag_ok = True
for (lat, lon) in test_pixels:
    p = full[(full.latitude == lat) & (full.longitude == lon)].sort_values("time").reset_index(drop=True)
    lhs = p["log_chl_lag1"].iloc[1:].values
    rhs = p["log_chl_t0"].iloc[:-1].values
    if not np.allclose(lhs, rhs, equal_nan=True):
        lag_ok = False
results["lag direction (lag1[t] == t0[t-1])"] = lag_ok

# --- 3) NaN preservation in lag columns ---
lag_nan_frac = sample["log_chl_lag1"].isna().mean()
results["NaNs survive in lag columns (no silent fill)"] = lag_nan_frac > 0

# --- 4) gap_age contract on the sample ---
ga_on_valid = sample.loc[sample["log_chl_t0"].notna(), "gap_age"]
results["gap_age == 0 where log_chl_t0 is valid"] = (ga_on_valid == 0).all()

# --- 5) every horizon has some valid targets ---
results["every target_h has some non-NaN"] = all(
    sample[f"target_h{h}"].notna().any() for h in HORIZONS
)

for name, ok in results.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

print(f"\nlag1 NaN fraction          : {lag_nan_frac*100:.1f}%")
print(f"target_h1 NaN fraction      : {sample['target_h1'].isna().mean()*100:.1f}%")
print(f"target_h7 NaN fraction      : {sample['target_h7'].isna().mean()*100:.1f}%")
print(f"Pixels tested for alignment : {len(test_pixels)}")

if all(results.values()):
    print("\nGate 4 PASSED — proceed to 05-Baselines.")
    print("  No leakage. Lags go backward, targets go forward, NaNs are honest.")
else:
    print("\nGate 4 FAILED — DO NOT continue. Fix the misalignment first.")
    print("  Common causes:")
    print("    • lag and target signs flipped (shift(time=-lag) instead of +lag)")
    print("    • predictor cube not reindexed onto the target time axis")
    print("    • spatial coords mis-named (lat vs latitude) causing silent broadcast")

PASS  per-pixel target alignment (all horizons)
PASS  lag direction (lag1[t] == t0[t-1])
PASS  NaNs survive in lag columns (no silent fill)
PASS  gap_age == 0 where log_chl_t0 is valid
PASS  every target_h has some non-NaN

lag1 NaN fraction          : 32.4%
target_h1 NaN fraction      : 32.7%
target_h7 NaN fraction      : 41.1%
Pixels tested for alignment : 3

Gate 4 PASSED — proceed to 05-Baselines.
  No leakage. Lags go backward, targets go forward, NaNs are honest.


## 4.8 — Persist the feature list

Stage 6 (XGBoost) and Stage 7 (SHAP) need to know which columns are features vs targets vs metadata. Save the list once so we're consistent across notebooks.

In [11]:
import json

META_COLS    = ["time", "latitude", "longitude", "month"]
TARGET_COLS  = [f"target_h{h}" for h in HORIZONS]
FEATURE_COLS = [c for c in sample.columns if c not in META_COLS + TARGET_COLS]

manifest = {
    "meta_cols": META_COLS,
    "target_cols": TARGET_COLS,
    "feature_cols": FEATURE_COLS,
    "horizons": HORIZONS,
    "lags": LAGS,
    "roll_window": ROLL_WIN,
    "use_3day_target": USE_3DAY_TARGET,
    "n_rows_sample": len(sample),
}
with open(PROC_DIR / "feature_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Features ({len(FEATURE_COLS)}):")
for c in FEATURE_COLS:
    print(f"  {c}")
print(f"\nTargets ({len(TARGET_COLS)}): {TARGET_COLS}")
print(f"\nManifest → {PROC_DIR / 'feature_manifest.json'}")

Features (15):
  log_chl_t0
  log_chl_lag1
  log_chl_lag2
  log_chl_lag3
  log_chl_lag7
  log_chl_lag14
  log_chl_roll7_mean
  log_chl_roll7_std
  sst_t0
  sst_lag1
  sst_roll7_mean
  depth_m
  doy_sin
  doy_cos
  gap_age

Targets (7): ['target_h1', 'target_h2', 'target_h3', 'target_h4', 'target_h5', 'target_h6', 'target_h7']

Manifest → data\processed\feature_manifest.json


---
## Next: `05-Baselines.ipynb`

Computes the **persistence** and **climatology** baselines — the bars every model in Stage 6 onwards must clear. PLAN.md: *"A fancy model that can't beat persistence is worthless, and you can only know that if you compute persistence first."*